In [0]:
-- Como é o histórico de transacoes diárias? E Mensal? E anual?

SELECT date(DtCriacao) as dtDia,
       count(*) AS qtdeTransacoes,
       count(distinct idCliente) AS qtdeCliente

FROM workspace.tmw_loyalty.transacoes

GROUP BY ALL
ORDER BY dtDia

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT date_trunc('MONTH',DtCriacao) as dtMes,
       count(*) AS qtdeTransacoes,
       count(distinct idCliente) AS qtdeCliente

FROM workspace.tmw_loyalty.transacoes

GROUP BY ALL
ORDER BY dtMes

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
WITH tb_daily AS (

    SELECT DATE(DtCriacao) dtAtivacao,
           IdCliente,
           sum(CASE WHEN  QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS QtdePontos,
           count(*) AS qtdeTransacao
    FROM workspace.tmw_loyalty.transacoes
    GROUP BY ALL
),

tb_dau AS(

    SELECT dtAtivacao,
           COUNT(DISTINCT idcliente)  AS DAU

    FROM tb_daily
    GROUP BY ALL
    ORDER BY dtAtivacao

),

tb_mau AS (

    SELECT t1.dtativacao,
           t1.DAU,
           count(DISTINCT t2.idcliente)  AS MAU,
           sum(QtdePontos) / count(DISTINCT t2.idcliente) AS ARPU

    FROM tb_dau AS t1
    LEFT JOIN tb_daily AS t2
    ON t1.dtAtivacao >= t2.dtAtivacao
    AND t1.dtAtivacao - INTERVAL 28 DAYS < t2.dtativacao

    GROUP BY ALL
    ORDER BY t1.dtativacao
),

tb_wau AS (
    
    SELECT t1.*,
           count(DISTINCT t2.idcliente)  AS WAU

    FROM tb_mau AS t1
    LEFT JOIN tb_daily AS t2
    ON t1.dtAtivacao >= t2.dtAtivacao
    AND t1.dtAtivacao - INTERVAL 7 DAYS < t2.dtativacao

    GROUP BY ALL
    ORDER BY t1.dtativacao
),

tb_mau_users AS (

    SELECT DISTINCT t1.dtativacao,
           t2.idcliente

    FROM tb_dau AS t1
    LEFT JOIN tb_daily AS t2
    ON t1.dtAtivacao >= t2.dtAtivacao
    AND t1.dtAtivacao - INTERVAL 28 DAYS < t2.dtativacao

    ORDER BY t1.dtativacao, t2.idcliente

),

tb_churn AS (

    SELECT t1.dtAtivacao + INTERVAL 28 DAY AS dtChurn,
           count(distinct t1.idcliente) AS MAU,
           count(distinct t2.idcliente) AS MAUfuturo,
           count(distinct t2.idcliente) / count(distinct t1.idcliente) AS txRetencao,
           1 - count(distinct t2.idcliente) / count(distinct t1.idcliente) AS txChurn

    FROM tb_mau_users as t1
    LEFT JOIN tb_mau_users t2
    ON t1.dtativacao = t2.dtativacao - INTERVAL 28 DAY
    AND t1.idcliente = t2.idcliente

    GROUP BY ALL
    HAVING count(distinct t2.idcliente) > 0
    ORDER BY 1

)

SELECT t1.*,
        t2.txRetencao,
        t2.txChurn

FROM tb_wau AS t1

LEFT JOIN tb_churn AS t2
ON t1.dtativacao = t2.dtChurn
order by dtativacao

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
WITH tb_daily AS (
    SELECT DISTINCT
        date(dtCriacao) AS dtDia,
        IdCliente
    FROM workspace.tmw_loyalty.transacoes
),

tb_lag (

    SELECT *,
            lag(dtDia) OVER (PARTITION BY idcliente order by dtDia) AS lagDt
    FROM tb_daily

),

tb_cliente AS (

    SELECT idcliente,
        avg(date_diff(dtDia, lagDt)) AS diaRecorrencia

    FROM tb_lag
    WHERE lagDt IS NOT NULL
    GROUP BY ALL

),

tb_qtde AS (

    SELECT diaRecorrencia,
            COUNT(DISTINCT idCliente) AS qtde

    FROM tb_cliente
    group by all
    order by diaRecorrencia

),

tb_acum AS (
    SELECT *,
           SUM(qtde) OVER (ORDER BY diaRecorrencia) AS qtdeAcum
    FROM tb_qtde
)

SELECT *,
       qtdeAcum/(select max(qtdeAcum) FROM tb_acum) AS pctAcum,
       1 - qtdeAcum/(select max(qtdeAcum) FROM tb_acum) AS pctSurvival
FROM tb_acum

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.